# Automatically scrape product data from Tiki API and generate SQL seeding file

This notebook scrapes products from Tiki for all 10 subcategories, generates random auction data, and saves it to `product.insert.sql`.

In [13]:
import os
import json
import random
import re
import requests
from datetime import datetime, timedelta
from dotenv import load_dotenv

load_dotenv()
print("✅ Libraries imported")

✅ Libraries imported


In [14]:
TIKI_CATEGORY_URL = "https://tiki.vn/api/personalish/v1/blocks/listings"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

# Map DB category names to Tiki API category IDs
DB_CAT_TO_TIKI = {
    "Thời Trang Nam": 925,
    "Thời Trang Nữ": 5404,
    "Giày Dép": 27572,
    "Điện Thoại & Máy Tính Bảng": 1795,
    "Laptop & PC": 8095,
    "Phụ Kiện Số": 1815,
    "Máy Ảnh & Quay Phim": 1801,
    "Tranh Ảnh": 27224,
    "Tượng & Điêu Khắc": 23102,
    "Trang Sức": 8374
}

def generate_times():
    # Generate random start and end times
    now = datetime.now()
    min_time = now - timedelta(days=7)
    max_time = now + timedelta(days=365)
    total_seconds = int((max_time - min_time).total_seconds())
    sec1 = random.randint(0, total_seconds)
    sec2 = random.randint(0, total_seconds)
    if sec1 == sec2:
        sec2 += 3600
    t1 = min_time + timedelta(seconds=sec1)
    t2 = min_time + timedelta(seconds=sec2)
    return min(t1, t2), max(t1, t2)

def generate_prices(base_price):
    if not base_price or base_price <= 0:
        base_price = random.randint(50, 500) * 10000
    start_price = int(base_price * random.uniform(0.6, 0.9))
    step_price = int(start_price * 0.05)
    buy_now_price = int(start_price * 1.5)
    return {
        'start_price': start_price,
        'current_price': start_price,
        'step_price': step_price,
        'buy_now_price': buy_now_price
    }

In [15]:
db_categories = []
# Resolve relative path to category.insert.sql
sql_paths = [
    "../../category/category.insert.sql",
    "../category/category.insert.sql",
    "data/category/category.insert.sql",
    "category.insert.sql"
]
for path in sql_paths:
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            content = f.read()
        matches = re.findall(r"VALUES\s*\(\s*(\d+)\s*,\s*'([^']+)'\s*,\s*([^,]+)", content, re.IGNORECASE)
        for cat_id, name, parent_id in matches:
            if parent_id.strip().upper() != 'NULL':
                db_categories.append((int(cat_id), name))
        if db_categories:
            print(f"✅ Loaded {len(db_categories)} subcategories from {path}")
            break
if not db_categories:
    db_categories = [
        (4, "Thời Trang Nam"),
        (5, "Thời Trang Nữ"),
        (6, "Giày Dép"),
        (7, "Điện Thoại & Máy Tính Bảng"),
        (8, "Laptop & PC"),
        (9, "Phụ Kiện Số"),
        (10, "Máy Ảnh & Quay Phim"),
        (11, "Tranh Ảnh"),
        (12, "Tượng & Điêu Khắc"),
        (13, "Trang Sức")
    ]
    print("✅ Using hardcoded default subcategories fallback")

seller_ids = [1, 2, 3]
products_data = []
print(f"Starting scraping data for {len(db_categories)} subcategories...")

for cat_id, cat_name in db_categories:
    tiki_cat_id = DB_CAT_TO_TIKI.get(cat_name, 1815)
    # Random count of products to scrape
    target_count = random.randint(20, 50)
    print(f"-> Fetching {target_count} products for '{cat_name}' (Tiki ID: {tiki_cat_id})")
    
    try:
        response = requests.get(
            TIKI_CATEGORY_URL,
            headers=headers,
            params={'limit': target_count, 'category': tiki_cat_id, 'page': 1}
        )
        
        if response.status_code == 200:
            tiki_products = response.json().get('data', [])
            for product in tiki_products:
                name = product.get('name', '')
                thumbnail_url = product.get('thumbnail_url', '')
                price = product.get('price', 0)
                name_escaped = name.replace("'", "''")
                # Generate English description text for products
                desc = f"High-quality product {name_escaped}. Genuine, reputable, and quality guaranteed."
                images = [thumbnail_url, thumbnail_url, thumbnail_url]
                images_array = "ARRAY[" + ", ".join([f"'{img}'" for img in images]) + "]"
                
                prices = generate_prices(price)
                start_time, end_time = generate_times()
                seller_id = random.choice(seller_ids)
                auto_ext = str(random.choice([True, False])).lower()
                
                sql = f"""INSERT INTO products (product_name, seller_id, step_price, start_price, current_price, buy_now_price, start_time, end_time, cat2_id, description, product_images, auto_extended)
VALUES ('{name_escaped}', {seller_id}, {prices['step_price']}, {prices['start_price']}, {prices['current_price']}, {prices['buy_now_price']}, '{start_time.isoformat()}', '{end_time.isoformat()}', {cat_id}, '{desc}', {images_array}, {auto_ext});"""
                products_data.append(sql)
        else:
            print(f"   ❌ Tiki API error: {response.status_code}")
    except Exception as e:
        print(f"   ❌ Scraping process error: {e}")

# Write generated inserts to file
with open('product.insert.sql', 'w', encoding='utf-8') as f:
    f.write('\n'.join(products_data))

print(f"\n🎉 Successfully generated product.insert.sql with {len(products_data)} products!")

✅ Loaded 10 subcategories from ../../category/category.insert.sql
Starting scraping data for 10 subcategories...
-> Fetching 44 products for 'Thời Trang Nam' (Tiki ID: 925)
-> Fetching 29 products for 'Thời Trang Nữ' (Tiki ID: 5404)
-> Fetching 41 products for 'Giày Dép' (Tiki ID: 27572)
-> Fetching 39 products for 'Điện Thoại & Máy Tính Bảng' (Tiki ID: 1795)
-> Fetching 35 products for 'Laptop & PC' (Tiki ID: 8095)
-> Fetching 28 products for 'Phụ Kiện Số' (Tiki ID: 1815)
-> Fetching 48 products for 'Máy Ảnh & Quay Phim' (Tiki ID: 1801)
-> Fetching 35 products for 'Tranh Ảnh' (Tiki ID: 27224)
-> Fetching 47 products for 'Tượng & Điêu Khắc' (Tiki ID: 23102)
-> Fetching 29 products for 'Trang Sức' (Tiki ID: 8374)

🎉 Successfully generated product.insert.sql with 333 products!
